In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split
import numpy as np
import os
import io
import matplotlib.pyplot as plt

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18
# install via: pip install huggingface_hub
from huggingface_hub import hf_hub_download

def create_resnet18_for_9_channels_bigearthnet_hf(num_classes=5):
    """
    Loads ResNet-18 pretrained on BigEarthNet v2.0 (all Sentinel bands),
    adapts for 9-channel input and `num_classes`.
    """
    # 1) Instantiate without any weights
    model = resnet18(weights=None)

    # 2) Download & load the BigEarthNet v2.0 "all bands" weights
    repo_id = "BIFOLD-BigEarthNetv2-0/resnet18-all-v0.2.0"
    blob = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")
    state_dict = torch.load(blob, map_location="cpu")
    # load, allowing missing keys for fc and conv1
    model.load_state_dict(state_dict, strict=False)

    # 3) Replace first conv layer to accept 9 channels instead of 3
    orig = model.conv1
    new_conv1 = nn.Conv2d(
        in_channels=9,
        out_channels=orig.out_channels,
        kernel_size=orig.kernel_size,
        stride=orig.stride,
        padding=orig.padding,
        bias=(orig.bias is not None)
    )
    with torch.no_grad():
        averaged_weights = orig.weight.mean(dim=1, keepdim=True)
        new_conv1.weight.copy_(averaged_weights.repeat(1, 9, 1, 1))
        if orig.bias is not None:
            new_conv1.bias.copy_(orig.bias)
    model.conv1 = new_conv1

    # 4) Replace the classifier for `num_classes`
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model

In [ ]:
# --- Helper function and Dataset Class ---
def get_initial_records_and_map(db_url_str):
    engine = create_engine(db_url_str)
    all_records_from_db, label_map, train_records_list, val_records_list = [], {}, [], []
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT id, surface_group FROM final_balanced_trail_segments WHERE image_data IS NOT NULL AND surface_group IS NOT NULL"))
            all_records_from_db = result.fetchall()
        if not all_records_from_db: return [], [], {}
        unique_labels = sorted(list(set(r[1] for r in all_records_from_db)))
        label_map = {label: idx for idx, label in enumerate(unique_labels)}
        all_records_with_labels = [(r[0], label_map[r[1]]) for r in all_records_from_db]
        train_records_list, val_records_list = train_test_split(
            all_records_with_labels, test_size=0.2, random_state=42,
            stratify=[r[1] for r in all_records_with_labels] if len(all_records_with_labels) > 1 else None
        )
        print(f"Pre-fetched data: {len(train_records_list)} train, {len(val_records_list)} val records. Label map size: {len(label_map)}")
    except Exception as e: print(f"Error during initial data fetch: {e}")
    finally:
        if engine: engine.dispose()
    return train_records_list, val_records_list, label_map

class TrailSegmentDataset(Dataset):
    def __init__(self, db_url, records_list, label_map, transform=None, for_stats_calc=False):
        self.db_url, self.engine, self.records, self.label_map, self.transform, self.for_stats_calc = \
            db_url, None, records_list, label_map, transform, for_stats_calc
    def _get_engine(self):
        if self.engine is None: self.engine = create_engine(self.db_url)
        return self.engine
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        seg_id, label = self.records[idx]
        try:
            with self._get_engine().connect() as conn:
                res = conn.execute(text("SELECT image_data FROM final_balanced_trail_segments WHERE id = :id"), {"id": seg_id})
                image_bytes = res.scalar()
            if image_bytes is None: return (None, label) if self.for_stats_calc else (_ for _ in ()).throw(ValueError(f"Img None for id {seg_id}"))
            image_array = np.load(io.BytesIO(image_bytes)).astype(np.float32)
            return (self.transform(image_array) if self.transform else image_array), label
        except Exception as e:
            print(f"Error in __getitem__ for seg_id {seg_id}: {e}")
            return (None, label) if self.for_stats_calc else (_ for _ in ()).throw(e)


In [ ]:
# --- Main Data Preparation ---
db_url = os.getenv('DATABASE_URL', "postgresql://markfinlay:@localhost:5432/osm")

train_img_records, val_img_records, global_label_map = get_initial_records_and_map(db_url)

# --- Normalization Calculation ---
print("Calculating mean and std for normalization...")
temp_train_dataset = TrailSegmentDataset(db_url, train_img_records, global_label_map, for_stats_calc=True)
all_imgs = [img for img, _ in temp_train_dataset if img is not None]
all_imgs_np = np.stack(all_imgs, axis=0)
channel_means = np.mean(all_imgs_np, axis=(0, 1, 2))
channel_stds = np.std(all_imgs_np, axis=(0, 1, 2))
channel_stds = np.maximum(channel_stds, 1e-6)
print(f"Means: {channel_means.tolist()}\nStds: {channel_stds.tolist()}")

# --- Transforms and DataLoaders ---
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=channel_means.tolist(), std=channel_stds.tolist()),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=channel_means.tolist(), std=channel_stds.tolist()),
])

train_dataset = TrailSegmentDataset(db_url, train_img_records, global_label_map, transform=train_transform)
val_dataset = TrailSegmentDataset(db_url, val_img_records, global_label_map, transform=val_transform)

num_workers = 0
pin_memory = False
if torch.cuda.is_available(): pin_memory = True

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=num_workers, pin_memory=pin_memory) if val_dataset else None

In [ ]:
# --- Setup for Training ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_classes = len(global_label_map)
model = create_resnet18_for_9_channels_bigearthnet_hf(num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Class weights
from collections import Counter
train_labels = [label for _, label in train_img_records]
counts = Counter(train_labels)
total = sum(counts.values())
weights = torch.tensor([total / (num_classes * counts.get(i, 1)) for i in range(num_classes)], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

# Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

# --- Training Loop ---
epoch_count = 30
best_val_accuracy = 0.0
patience_early_stop = 8
patience_counter = 0
model_save_path = "best_resnet18_model_bigearthnet.pth"

print("--- Starting ResNet18 Training ---")
for epoch in range(epoch_count):
    model.train()
    running_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)
    
    epoch_loss = running_loss / train_total
    print(f"Epoch [{epoch+1}/{epoch_count}] - Train Loss: {epoch_loss:.4f}, Train Acc: {(100*train_correct/train_total):.2f}%")

    if val_loader:
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item() * images.size(0)
                val_correct += (outputs.argmax(dim=1) == labels).sum().item()
                val_total += labels.size(0)
        
        epoch_val_loss = val_loss / val_total
        epoch_val_accuracy = 100 * val_correct / val_total
        print(f"→ Validation Loss: {epoch_val_loss:.4f}, Validation Acc: {epoch_val_accuracy:.2f}%")
        
        # Scheduler step
        scheduler.step(epoch_val_loss)

        if epoch_val_accuracy > best_val_accuracy:
            best_val_accuracy = epoch_val_accuracy
            patience_counter = 0
            torch.save(model.state_dict(), model_save_path)
            print(f"✓ Saved new best model to {model_save_path} with accuracy: {best_val_accuracy:.2f}%")
        else:
            patience_counter += 1
        
        if patience_counter >= patience_early_stop:
            print(f"Early stopping triggered after {patience_early_stop} epochs of no improvement.")
            break
print("--- Training complete ---")

In [ ]:
# --- Main Analysis Script ---

# 1. SETUP: Define paths and device
model_path = "best_resnet18_model_bigearthnet ????.pth"
db_url = "postgresql://markfinlay:@localhost:5432/osm"
device = torch.device("mps")
print(f"Using device: {device}")
_, val_img_records, global_label_map = get_initial_records_and_map(db_url)

# We need the normalization statistics from training. 
# For simplicity in this script, we'll recalculate them from the training set.
# In a production pipeline, you would save these stats and load them here.
print("Recalculating normalization stats from the training set (for transform consistency)...")
train_img_records, _, _ = get_initial_records_and_map(db_url)
temp_train_dataset = TrailSegmentDataset(db_url, train_img_records, global_label_map)
all_imgs = [img for img, _ in temp_train_dataset if img is not None]

all_imgs_np = np.stack(all_imgs, axis=0)
channel_means = np.mean(all_imgs_np, axis=(0, 1, 2))
channel_stds = np.std(all_imgs_np, axis=(0, 1, 2))
channel_stds = np.maximum(channel_stds, 1e-6)
print("Normalization stats calculated.")

# Create the validation transform (must match what was used in training)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=channel_means.tolist(), std=channel_stds.tolist()),
])

# Create the validation dataset and loader
val_dataset = TrailSegmentDataset(db_url, val_img_records, global_label_map, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(global_label_map)
model = create_resnet18_for_9_channels_bigearthnet_hf(num_classes)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval() # IMPORTANT: Set model to evaluation mode
print(f"Model '{model_path}' loaded successfully.")

# 4. GET PREDICTIONS: Iterate through validation set and collect predictions
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import os
import io
import matplotlib.pyplot as plt
import seaborn as sns

# 5. COMPUTE AND PLOT CONFUSION MATRIX
print("Generating confusion matrix...")
cm = confusion_matrix(all_labels, all_preds)

# Get class names from the label map for plotting
class_names = [item[0] for item in sorted(global_label_map.items(), key=lambda item: item[1])]

# Create a figure for the plot
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 10))

# Create a heatmap using seaborn
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=ax, xticklabels=class_names, yticklabels=class_names)

# Add annotations (counts and percentages) to each cell
for i in range(len(class_names)):
    for j in range(len(class_names)):
        count = cm[i, j]
        percentage = f"{(count / np.sum(cm[i]))*100:0.1f}%" if np.sum(cm[i]) > 0 else "0.0%"
        text = f"{count}\n{percentage}"
        ax.text(j + 0.5, i + 0.5, text, ha='center', va='center', color='black', fontsize=12)

# Set labels and title
ax.set_xlabel('Predicted Label', fontsize=14, labelpad=10)
ax.set_ylabel('True Label', fontsize=14, labelpad=10)
ax.set_title('Confusion Matrix for ResNet18 Model', fontsize=16, pad=20)

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# Save the figure
output_filename = "confusion_matrix_resnet18.png"
plt.savefig(output_filename, dpi=300)
print(f"Confusion matrix plot saved to '{output_filename}'")

plt.show()